# Feature Engineering: Reading Log Dataset (Pandas)

This notebook performs data quality checks and feature engineering on the `reading_log.parquet` dataset using **Pandas**.

## Schema (Input)
- **work_key** (String, nullable)
- **edition_key** (String, nullable)
- **status** (String, nullable)
- **log_date** (Date32, nullable)

## Objectives
1. Assess data quality (null values, status values, date validity)
2. Validate `work_key` (required for joins; must start with `/works/`)
3. Standardize `status` (lowercase, trim spaces, map variations)
4. Validate `log_date` (remove future dates)
5. Derive `log_year` from `log_date` for temporal analysis
6. Generate quality report

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuration
DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

## Step 1: Load and Inspect Raw Data

In [2]:
# Load reading log data
input_path = PROCESSED_DIR / 'reading_log.parquet'
df = pd.read_parquet(input_path)

print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSchema:")
print(df.dtypes)
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Total rows: 11,529,486
Columns: ['work_key', 'edition_key', 'status', 'log_date']

Schema:
work_key          str
edition_key       str
status            str
log_date       object
dtype: object

Memory usage: 1146.97 MB


In [3]:
# Display first few rows
print(df.head(10))
print(df.tail(10))

             work_key         edition_key        status    log_date
0   /works/OL4439701W                      Already Read  2017-12-11
1     /works/OL63060W   /books/OL5816906M  Already Read  2017-12-26
2  /works/OL10417330W                      Want to Read  2017-11-08
3   /works/OL4466500W   /books/OL2712504M  Want to Read  2017-11-08
4   /works/OL5920528W                      Want to Read  2018-01-05
5  /works/OL15852999W                      Want to Read  2018-03-15
6   /works/OL3743769W   /books/OL4114449M  Already Read  2017-11-10
7   /works/OL4619760W  /books/OL16479114M  Want to Read  2018-02-06
8   /works/OL3603559W   /books/OL3548301M  Want to Read  2017-12-29
9   /works/OL6306701W                      Already Read  2017-11-10
                    work_key         edition_key             status  \
11529476   /works/OL8801171W                           Already Read   
11529477    /works/OL112578W                      Currently Reading   
11529478    /works/OL439771W           

## Step 2: Data Quality Assessment

In [4]:
# Check null values
print("=== Null Value Counts ===")
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df)) * 100

quality_df = pd.DataFrame({
    'Column': null_counts.index,
    'Null Count': null_counts.values,
    'Null Percentage': null_pct.values
})
print(quality_df.to_string(index=False))

=== Null Value Counts ===
     Column  Null Count  Null Percentage
   work_key           0              0.0
edition_key           0              0.0
     status           0              0.0
   log_date           0              0.0


In [5]:
# Status value distribution
print("=== Status Value Distribution ===")
status_counts = df['status'].value_counts(dropna=False)
print(status_counts)
print(f"\nUnique status values: {df['status'].nunique()}")

=== Status Value Distribution ===
status
Want to Read         9802870
Already Read         1166475
Currently Reading     560141
Name: count, dtype: int64

Unique status values: 3


In [6]:
# log_date analysis (use .date() so types match Parquet date)
print("=== log_date Analysis ===")
dates_non_null = df['log_date'].dropna()
if len(dates_non_null) > 0:
    print(f"Non-null log_date count: {len(dates_non_null):,}")
    print(f"Min date: {dates_non_null.min()}")
    print(f"Max date: {dates_non_null.max()}")
    today = pd.Timestamp.now().date()
    future = (dates_non_null > today).sum()
    print(f"Future dates (after today): {future:,}")
    old_cutoff = pd.Timestamp('1900-01-01').date()
    very_old = (dates_non_null < old_cutoff).sum()
    print(f"Dates before 1900: {very_old:,}")

=== log_date Analysis ===
Non-null log_date count: 11,529,486
Min date: 2017-10-27
Max date: 2025-12-31
Future dates (after today): 0
Dates before 1900: 0


In [7]:
# work_key validation: required for joins, should start with /works/
print("=== work_key Validation ===")
work_key_non_null = df['work_key'].dropna()
valid_work_key = work_key_non_null.astype(str).str.startswith('/works/', na=False)
print(f"Rows with non-null work_key: {len(work_key_non_null):,}")
print(f"Rows with work_key starting with /works/: {valid_work_key.sum():,}")
print(f"Rows with null or malformed work_key: {len(df) - valid_work_key.sum():,}")

malformed = work_key_non_null[~valid_work_key]
if len(malformed) > 0:
    print(f"\nSample malformed work_key values:")
    for val in malformed.head(5):
        print(f"  '{val}'")

=== work_key Validation ===
Rows with non-null work_key: 11,529,486
Rows with work_key starting with /works/: 11,529,486
Rows with null or malformed work_key: 0


## Step 3: Clean and Transform Data

In [8]:
df_cleaned = df.copy()
print(f"Original row count: {len(df_cleaned):,}")

Original row count: 11,529,486


In [9]:
# Keep only rows with valid work_key (required for joins)
before_filter = len(df_cleaned)
df_cleaned = df_cleaned[
    df_cleaned['work_key'].notna() &
    (df_cleaned['work_key'].astype(str).str.strip() != '') &
    df_cleaned['work_key'].astype(str).str.startswith('/works/', na=False)
].copy()
rows_removed = before_filter - len(df_cleaned)
print(f"Rows removed (null/invalid work_key): {rows_removed:,}")
print(f"Rows retained: {len(df_cleaned):,}")

Rows removed (null/invalid work_key): 0
Rows retained: 11,529,486


In [10]:
# Standardize status: lowercase, strip whitespace
print("Standardizing status...")
df_cleaned['status'] = df_cleaned['status'].astype(str).str.strip().str.lower()
# Map empty string or 'nan' back to null
df_cleaned.loc[df_cleaned['status'].isin(['', 'nan']), 'status'] = np.nan
print(f"Unique status values after cleaning: {df_cleaned['status'].dropna().nunique()}")
print(df_cleaned['status'].value_counts(dropna=False).head(15))

Standardizing status...
Unique status values after cleaning: 3
status
want to read         9802870
already read         1166475
currently reading     560141
Name: count, dtype: int64


In [11]:
# Derive log_year from log_date (for temporal analysis)
print("Extracting log_year from log_date...")
df_cleaned['log_year'] = pd.to_datetime(df_cleaned['log_date'], errors='coerce').dt.year
df_cleaned['log_year'] = df_cleaned['log_year'].astype('Int16')
valid_year = df_cleaned['log_year'].notna().sum()
print(f"Rows with valid log_year: {valid_year:,}")
print(f"Rows with null log_year: {df_cleaned['log_year'].isna().sum():,}")

Extracting log_year from log_date...
Rows with valid log_year: 11,529,486
Rows with null log_year: 0


In [12]:
# Remove future log_date (use .date() to match Parquet date type)
today = pd.Timestamp.now().date()
before_future = len(df_cleaned)
df_cleaned = df_cleaned[
    df_cleaned['log_date'].isna() | (df_cleaned['log_date'] <= today)
].copy()
future_removed = before_future - len(df_cleaned)
print(f"Rows with future log_date removed: {future_removed:,}")
print(f"Rows retained: {len(df_cleaned):,}")

Rows with future log_date removed: 0
Rows retained: 11,529,486


In [13]:
# Final columns: work_key, edition_key, status, log_date, log_year
df_cleaned = df_cleaned[["work_key", "edition_key", "status", "log_date", "log_year"]].copy()

print("Final schema:")
print(df_cleaned.dtypes)
print(f"\nFinal row count: {len(df_cleaned):,}")

Final schema:
work_key          str
edition_key       str
status            str
log_date       object
log_year        Int16
dtype: object

Final row count: 11,529,486


## Step 4: Final Quality Check

In [14]:
print("=== Final Quality Check ===")
valid_log_year = df_cleaned['log_year'].notna().sum()
pct_valid = (valid_log_year / len(df_cleaned)) * 100 if len(df_cleaned) > 0 else 0
print(f"Rows with valid log_year: {valid_log_year:,} ({pct_valid:.2f}%)")
print(f"Rows with null log_year: {len(df_cleaned) - valid_log_year:,}")
if valid_log_year > 0:
    print(f"\nlog_year - Min: {df_cleaned['log_year'].min()}, Max: {df_cleaned['log_year'].max()}")
print(f"\nStatus distribution:")
print(df_cleaned['status'].value_counts(dropna=False).head(10))

=== Final Quality Check ===
Rows with valid log_year: 11,529,486 (100.00%)
Rows with null log_year: 0

log_year - Min: 2017, Max: 2025

Status distribution:
status
want to read         9802870
already read         1166475
currently reading     560141
Name: count, dtype: int64


In [15]:
print("\nSample of cleaned data:")
df_cleaned.head(20)


Sample of cleaned data:


,work_key,edition_key,status,log_date,log_year
0,/works/OL4439701W,,already read,2017-12-11,2017
1,/works/OL63060W,/books/OL5816906M,already read,2017-12-26,2017
2,/works/OL10417330W,,want to read,2017-11-08,2017
3,/works/OL4466500W,/books/OL2712504M,want to read,2017-11-08,2017
4,/works/OL5920528W,,want to read,2018-01-05,2018
5,/works/OL15852999W,,want to read,2018-03-15,2018
6,/works/OL3743769W,/books/OL4114449M,already read,2017-11-10,2017
7,/works/OL4619760W,/books/OL16479114M,want to read,2018-02-06,2018
8,/works/OL3603559W,/books/OL3548301M,want to read,2017-12-29,2017
9,/works/OL6306701W,,already read,2017-11-10,2017


## Step 5: Save Cleaned Data

In [16]:
output_path = PROCESSED_DIR / 'reading_log_cleaned.parquet'
df_cleaned.to_parquet(output_path, index=False)
print(f"Saved cleaned data to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024**2:.2f} MB")

Saved cleaned data to ../data/processed/reading_log_cleaned.parquet
File size: 162.32 MB


## Step 6: Generate Quality Report

In [17]:
report_path = REPORTS_DIR / 'data_quality_reading_log_pandas.md'
valid_count = df_cleaned['log_year'].notna().sum()
null_year_count = df_cleaned['log_year'].isna().sum()

report = f"""# Data Quality Report: Reading Log Dataset (Pandas)

## Summary
- **Original row count**: {len(df):,}
- **Cleaned row count**: {len(df_cleaned):,}
- **Rows removed**: {len(df) - len(df_cleaned):,} ({(len(df) - len(df_cleaned))/len(df)*100:.2f}%)
- **Rows retained**: {len(df_cleaned)/len(df)*100:.2f}%

## Data Quality Metrics

### Log Year (derived from log_date)
- **Rows with valid log_year**: {valid_count:,} ({valid_count/len(df_cleaned)*100:.2f}%)
- **Rows with null log_year**: {null_year_count:,} ({null_year_count/len(df_cleaned)*100:.2f}%)

"""
if valid_count > 0:
    report += f"""### Log Year Statistics
- **Minimum year**: {df_cleaned['log_year'].min()}
- **Maximum year**: {df_cleaned['log_year'].max()}

"""
report += """## Schema Changes
- **Added**: `log_year` (Int16, nullable) - derived from log_date.year
- **Standardized**: `status` (lowercase, trimmed)

## Cleaning Steps Applied
1. Removed rows with null or invalid work_key (must start with /works/)
2. Standardized status (lowercase, strip whitespace)
3. Derived log_year from log_date for temporal analysis
4. Removed rows with future log_date
"""
report_path.write_text(report)
print(f"Quality report saved to {report_path}")
print("\n" + report_path.read_text())

Quality report saved to ../reports/data_quality_reading_log_pandas.md

# Data Quality Report: Reading Log Dataset (Pandas)

## Summary
- **Original row count**: 11,529,486
- **Cleaned row count**: 11,529,486
- **Rows removed**: 0 (0.00%)
- **Rows retained**: 100.00%

## Data Quality Metrics

### Log Year (derived from log_date)
- **Rows with valid log_year**: 11,529,486 (100.00%)
- **Rows with null log_year**: 0 (0.00%)

### Log Year Statistics
- **Minimum year**: 2017
- **Maximum year**: 2025

## Schema Changes
- **Added**: `log_year` (Int16, nullable) - derived from log_date.year
- **Standardized**: `status` (lowercase, trimmed)

## Cleaning Steps Applied
1. Removed rows with null or invalid work_key (must start with /works/)
2. Standardized status (lowercase, strip whitespace)
3. Derived log_year from log_date for temporal analysis
4. Removed rows with future log_date



## Summary

Done: load & inspect → quality assessment → work_key validation → status standardization → log_year derivation → date validation → save `reading_log_cleaned.parquet` and quality report.